In [ ]:
%%capture
# !pip install openai

In [1]:
import json
import random
from openai import OpenAI
from tqdm import tqdm
import random
from datasets import Dataset
# from google.colab import userdata
import os
import pickle

d:\Kuliah\SKRIPSI\code\playground\skripsi-playground-venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

# HF_TOKEN = userdata.get('HF_TOKEN')
# EMBEDDING_MODEL_ID = "google/embeddinggemma-300m"
PICKLE_BACKUP_PATH = "../playground/knowledge-base/medical_documents_enriched_v1.pkl"
OUTPUT_PATH = 'nutrition_qa_golden_dataset.json'

In [11]:
def generate_multi_context_qa(contexts, n_questions=3):
    combined_context = "\n\n---\n\n".join(contexts)

    prompt = f"""
      Anda bertugas membuat dataset evaluasi berkualitas tinggi untuk sistem RAG dengan topik rekomendasi medis.

      Gunakan HANYA informasi yang terdapat pada bagian "Konteks" di bawah ini.

      Instruksi:
      1. Buat {n_questions} pertanyaan multi-hop.
      2. Setiap pertanyaan WAJIB membutuhkan informasi dari lebih dari satu konteks.
      3. Jawaban HARUS dapat diturunkan secara eksplisit dari teks.
      4. Fokus HANYA pada bagian ISI LAPORAN (narasi, pembahasan, intervensi, monitoring, edukasi, rencana tindakan, evaluasi, rekomendasi, perkembangan pasien).
      5. DILARANG membuat pertanyaan tentang:
        - Diagnosis medis
        - Diagnosis gizi
        - Data antropometri (BB, TB, IMT, LLA, persentil, z-score)
        - Perhitungan matematis
      6. Jika informasi hanya berasal dari satu konteks, JANGAN gunakan.
      7. Jangan mengulang pertanyaan yang secara eksplisit menyalin kalimat teks.

      Keluaran WAJIB dalam format JSON berikut:

      [
        {{
          "question": "...",
          "answer": "...",
        }}
      ]

      Konteks:
      {combined_context}
    """


    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.3,
    )

    content = response.choices[0].message.content

    try:
        return json.loads(content)
    except:
        print("JSON parsing failed")
        return []


In [12]:
def build_multi_context_dataset(
    documents,
    group_size=2,
    max_rows=None
):
    dataset = []

    sampled_indices = random.sample(
        range(len(documents) - group_size),
        min(max_rows, len(documents) - group_size)
    )

    for i in tqdm(sampled_indices):

        if max_rows is not None and len(dataset) >= max_rows:
            break

        selected_docs = documents[i:i + group_size]
        selected_texts = [doc.page_content for doc in selected_docs]

        qa_pairs = generate_multi_context_qa(selected_texts)
        if isinstance(qa_pairs, dict) and "data" in qa_pairs:
          qa_list = qa_pairs["data"]
        elif isinstance(qa_pairs, list):
          qa_list = qa_pairs
        else:
          print("Format output tidak dikenali")
          continue

        for qa in qa_list:

            if max_rows is not None and len(dataset) >= max_rows:
                break

            dataset.append({
                "question": qa["question"],
                "ground_truth": qa["answer"],
                "gold_contexts": selected_texts,
                "chunk_metadata": [doc.metadata for doc in selected_docs],
            })

    return dataset


In [7]:
def save_dataset(dataset, path=OUTPUT_PATH):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(dataset)} samples to {path}")


In [14]:
with open(PICKLE_BACKUP_PATH, 'rb') as file:
    documents = pickle.load(file)

dataset = build_multi_context_dataset(documents, group_size=3, max_rows=150)
save_dataset(dataset)


  1%|▏         | 2/150 [00:17<20:43,  8.40s/it]

Format output tidak dikenali


 14%|█▍        | 21/150 [02:39<17:39,  8.21s/it]

Format output tidak dikenali


 17%|█▋        | 25/150 [03:07<14:48,  7.11s/it]

...

Format output tidak dikenali


 35%|███▍      | 52/150 [05:59<10:13,  6.26s/it]

Format output tidak dikenali


 41%|████▏     | 62/150 [07:11<10:12,  6.96s/it]

Saved 150 samples to nutrition_qa_golden_dataset.json


In [9]:
def convert_to_ragas_format(dataset):
    return Dataset.from_list([
        {
            "question": item["question"],
            "contexts": item["gold_contexts"],
            "ground_truth": item["ground_truth"],
        }
        for item in dataset
    ])